# 4 — Several graphs from one date, trained together, applied to another

One date is one graph, but `training` also takes a list. Here the training set is **four random
subsets of the 2018-01-01 grid points at stride `STRIDE`**, each a `Data` with its own nodes and
its own edges; the trained model is then applied to the **complete 2019-08-01 field at native
stride 1** — 1,035,360 locations of a date it never saw, on a grid finer than anything it trained
on.

An epoch is one optimizer step per graph, so `epochs=200` over four graphs is 800 steps; the
reported `train_loss`/`val_loss` are the means over those graphs.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import numpy as np
import torch

from nofe import NOFE, describe
from nofe.affinity import affinity_kl_loss
from nofe.search import NearestIndex

# Resolve paths from the repository layout, not the process cwd: VS Code always starts kernels in
# the workspace folder (`jupyter.notebookFileRoot` is ignored since 2024.4), while `_execute.py`
# runs from this directory. Walk up from the cwd looking for the repo root, so both agree.
REPO = next(
    (
        ancestor
        for ancestor in (Path.cwd(), *Path.cwd().parents)
        if (ancestor / "pyproject.toml").is_file()
    ),
    None,
)
if REPO is None:
    raise RuntimeError(f"run this notebook from inside the nofe checkout, not {Path.cwd()}")
ERA5 = REPO / "examples" / "era5"
DATA = ERA5 / "data"

# Both dates ship in Git beside this notebook, so a missing file means a partial checkout; name the
# path the loader would otherwise fail on opaquely.
if not (DATA / "2018-01-01.nc").exists():
    raise FileNotFoundError(
        f"missing {DATA / '2018-01-01.nc'}\n"
        f"The ERA5 inputs are tracked in {DATA}; restore them with `git checkout -- {DATA}`."
    )

# The notebooks import the ERA5 helpers straight from the example scripts, so the loader, the
# graph generator and the globe plot have exactly one implementation in the repo. They are
# imported after `sys.path` is extended, which is why the noqa markers below are needed: the ERA5
# demo lives in `scripts/`, its graph generator here in `utilities/`, and the shared graph helpers
# one level up in `examples/`.
sys.path.insert(0, str(ERA5 / "scripts"))
sys.path.insert(0, str(ERA5 / "utilities"))
sys.path.insert(0, str(REPO / "examples"))
import era5  # noqa: E402
from era5_graph import Era5Graph  # noqa: E402


def kl(data, embedding) -> float:
    """Mean KL between a graph's own feature affinities and an embedding of its rows."""
    return float(affinity_kl_loss(data.x, embedding, data.edge_index, num_nodes=data.num_nodes))


def pca_baseline(data, dims: int = 3) -> torch.Tensor:
    """Best linear projection to ``dims`` dimensions: the reference the model has to beat."""
    centred = data.x - data.x.mean(dim=0, keepdim=True)
    _, _, basis = torch.linalg.svd(centred, full_matrices=False)
    return centred @ basis[:dims].T


def neighbour_sets(points: torch.Tensor, k: int = 10) -> np.ndarray:
    """Exact ``k``-nearest-neighbour ids per row, self excluded, on the active backend."""
    values = points.detach().cpu().numpy().astype(np.float32)
    rows = np.arange(values.shape[0])
    sender, receiver = NearestIndex(values, rows).nearest(rows, k)
    # ``nearest`` returns pairs sorted by receiver, so each receiver's block is already
    # nearest-first and ``reshape`` places it on that row.
    return sender.reshape(values.shape[0], k)


def knn_agreement(reference: torch.Tensor, candidate: torch.Tensor, k: int = 10) -> float:
    """Mean share of exact k-nearest neighbours two point sets have in common."""
    ref = neighbour_sets(reference, k)
    cand = neighbour_sets(candidate, k)
    shared = [len(set(a) & set(b)) / k for a, b in zip(ref, cand, strict=True)]
    return float(np.mean(shared))


def close(a: torch.Tensor, b: torch.Tensor, rtol: float = 1e-4) -> bool:
    """Relative agreement in units of the embedding's own scale."""
    return bool((a - b).abs().max() <= rtol * a.abs().max())


print("nofe ready")


## Sample subgraphs from the first date

Each subgraph keeps a random `SAMPLE` of the grid points (`PointGraph(sample=...)`, drawn with its
own seed): same radius, fewer points, so a few percent of the kept nodes end up with no neighbour
inside their own subgraph. They are not broken — a node with no incoming edge is embedded from its
own features through the root weight — but they carry no neighbourhood information, which is why the
in-degree is printed rather than assumed.

In [ ]:
STRIDE = 8
# Chord on the unit sphere: a 2.5 degree arc, the same neighbourhood as notebook 1.
RADIUS = 0.04363
SAMPLE = 8_010  # half of the 16,020 stride-8 grid points
NUM_GRAPHS = 4

graph = Era5Graph(radius=RADIUS, stride=STRIDE, data_dir=DATA)
features, latitudes, longitudes = graph.load("2018-01-01")
print(f"{features.shape[0] * features.shape[1]} grid points on 2018-01-01, "
      f"{features.shape[2]} variables each")


def sample_graph(seed: int):
    """A random subset of the grid points, with a radius neighbourhood of its own."""
    builder = Era5Graph(radius=RADIUS, stride=STRIDE, data_dir=DATA, sample=SAMPLE, seed=seed)
    return builder.build_raster(features, latitudes, longitudes)


graphs = [sample_graph(seed) for seed in range(NUM_GRAPHS)]
for index, sampled in enumerate(graphs):
    degrees = torch.bincount(sampled.edge_index[1], minlength=sampled.num_nodes)
    print(
        f"graph {index}: {sampled.num_nodes} nodes, {sampled.edge_index.shape[1]} edges, "
        f"mean in-degree {degrees.float().mean():.2f}, isolated {int((degrees == 0).sum())}"
    )


## Train on the list

`training` runs one optimizer step per graph per epoch, returns one history per graph, and averages
the losses it reports. `loss` on a list returns one float per graph, so the list is the per-graph
scores below in order.

In [ ]:
model = NOFE(3, width=8, ker_width=32, nystrom=0.25)
histories = model.training(graphs, epochs=200, seed=0, verbose=True)
per_graph = model.loss(graphs)
print(f"{len(histories[0])} epochs, {len(histories[0]) * len(graphs)} optimizer steps")
print(f"graph 0 loss {histories[0][0]['train_loss']:.5f} -> {histories[0][-1]['train_loss']:.6f}")
print("mean KL per graph:", [round(value, 6) for value in per_graph])
assert len(histories) == len(per_graph) == NUM_GRAPHS


## Infer on the full second date, at native resolution

Training ran on 50% samples at `STRIDE`, because the objective needs every training edge resident
on every epoch. Inference does not: it is one forward pass, so it can cover the whole grid. The
model is therefore applied to **all 719 x 1440 = 1,035,360 grid points** of 2019-08-01 at stride 1,
a grid it has never seen.

The radius has to shrink with the stride — 2.5 degrees at stride 1 would be ~917M edges (~40 GB) —
and `0.005454` keeps the neighbourhood the fine grid can hold. `embed` stays single-graph and
returns one row per node.

In [ ]:
INFER_STRIDE = 1
INFER_RADIUS = 0.005454
full_graph = Era5Graph(radius=INFER_RADIUS, stride=INFER_STRIDE, data_dir=DATA)
full_features, full_latitudes, full_longitudes = full_graph.load("2019-08-01")
full = full_graph.build_raster(full_features, full_latitudes, full_longitudes)
print(f"full grid: {full.num_nodes} nodes, {full.edge_index.shape[1]} edges")

embedding = model.embed(full)
print("embedding on 2019-08-01:", tuple(embedding.shape))
assert embedding.shape == (full.num_nodes, 3)
assert torch.isfinite(embedding).all()


## Plot the embedding of the unseen date

R/G/B are the three embedding dimensions, one `pcolormesh` quad per grid point, scaled by the
5th/95th percentile spread (`era5.PERCENTILE`).

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

figure, axis = plt.subplots(
    1, 1, figsize=(12.0, 6.5), subplot_kw={"projection": ccrs.PlateCarree()}
)
axis.set_global()
era5.draw_globe(axis, full_latitudes, full_longitudes, embedding.numpy())
axis.coastlines(linewidth=0.5)
axis.set_title(f"2019-08-01, never trained on: {full.num_nodes} nodes")
plt.show()


## Takeaways

|Fact|Why it matters|
|---|---|
|`training` takes a list|One operator covers many graphs; an epoch is one step per graph|
|`sample=N` draws N rows|A subgraph is the same builder with fewer points, nothing else|
|`loss` on a list is per graph|Each graph counts once, whatever its node count|
|Inference needs no training grid|A coarse, sub-sampled model embeds the full-resolution field|
|`embed` takes one graph or a list|It returns the matching type: one tensor, or one per graph|